# LangGraph: Building Stateful Agent Graphs

LangGraph builds LLM applications as explicit state graphs — nodes are functions, edges are control flow. Good for interview questions on agent orchestration, ReAct loops, and why you'd reach for a graph instead of one-shot LLM calls.

**Covers:**
1. A minimal single-node graph (core `StateGraph` mechanics)
2. A tool-calling agent loop (conditional edges, `ToolNode`, the ReAct pattern)

In [2]:
!uv add langgraph langchain-openai python-dotenv

Resolved 126 packages in 472ms                                       
Prepared 5 packages in 227ms                                             
Installed 20 packages in 30ms                               
 + distro==1.9.0
 + jsonpatch==1.33
 + langchain-core==1.6.2
 + langchain-openai==1.6.2
 + langchain-protocol==0.0.19
 + langgraph==1.2.11
 + langgraph-checkpoint==4.2.0
 + langgraph-prebuilt==1.1.0
 + langgraph-sdk==0.4.4
 + langsmith==0.12.4
 + orjson==3.12.0
 + ormsgpack==1.12.2
 + regex==2026.9.10
 + requests-toolbelt==1.0.0
 + tenacity==9.1.4
 + tiktoken==0.14.0
 + uuid-utils==0.17.1
 + websockets==16.1.1
 + xxhash==4.0.1
 + zstandard==0.25.0


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [7]:
llm = ChatOpenAI(
    model="gpt-5.4"
)

In [8]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""
    return str(eval(expression))

In [9]:
calculator.invoke("25 * 17")

'425'

In [11]:
llm_with_tools = llm.bind_tools([calculator])

# This tells the LLM:

# You are allowed to request the calculator tool.

## 6. Define the State

Our state will contain the conversation:
```
State
 └── messages
      ├── User message
      ├── AI message
      ├── Tool message
      └── AI message
```

In [12]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [13]:
# 7. Create the Agent Node
def agent(state: State):

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }


The node does one thing:
```
State
  ↓
LLM
  ↓
New message
  ↓
State
```

8. Create the Tool Node

ToolNode knows how to:

```
receive tool call
      ↓
find the correct tool
      ↓
execute it
      ↓
return tool result
```

In [14]:
from langgraph.prebuilt import ToolNode

tools = [calculator]

tool_node = ToolNode(tools)

# 9. Create the Routing Logic

In [15]:
def should_continue(state: State):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END

This is the agent's decision point:

                 Agent
                   ↓
             Tool requested?
              /           \
            YES            NO
             ↓              ↓
           Tools           END

10. Build the Graph

In [22]:
graph = StateGraph(State)

graph.add_node("agent", agent)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    should_continue
)

graph.add_edge("tools", "agent")

```
START
  ↓
Agent
  ↓
Should continue?
  │
  ├── tools ──→ Tools
  │               │
  │               ↓
  │             Agent
  │
  └── END
  ```


## 11. Compile

In [25]:
app = graph.compile()


# compile() turns our graph definition into an executable graph.

In [26]:
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 25 * 17?"
        }
    ]
})

In [27]:
for message in result["messages"]:
    print(type(message).__name__, ":", message.content)

HumanMessage : What is 25 * 17?
AIMessage : 
ToolMessage : 425
AIMessage : 425


What you just built

You have created a real LangGraph agent:

                 User
                   ↓
              ┌────────┐
              │ Agent  │
              │  LLM   │
              └───┬────┘
                  ↓
            Tool required?
             /         \
           YES          NO
            ↓            ↓
       ┌────────┐       END
       │ Tools  │
       └───┬────┘
           ↓
         Agent
           ↓
         Answer

## 1. A Minimal Graph

The smallest possible LangGraph: one state shape, one node, wired `START -> node -> END`. This establishes the core vocabulary before adding LLM calls and branching.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

**State** is a `TypedDict` describing what flows between nodes — the shared payload passed along each edge.

In [3]:
class State(TypedDict):
    message: str

**Node** is just a plain function: takes the state, returns a (partial) dict that gets merged into it. No special base class or decorator required.

In [4]:
def hello_node(state: State):
    return {
        "message": "Hello " + state["message"]
    }

**Wiring the graph**: register nodes with `add_node`, connect them with `add_edge`, then `compile()` into a runnable `app`. `START` and `END` are sentinel markers for entry/exit.

In [5]:
graph = StateGraph(State)

graph.add_node("hello", hello_node)

graph.add_edge(START, "hello")
graph.add_edge("hello", END)

app = graph.compile()

Running the compiled graph is just `.invoke(initial_state)` — the same interface as a single LLM call, which is why graphs are easy to drop in wherever `.invoke()` is already used.

In [6]:
result = app.invoke({
    "message": "Surendra"
})

print(result)

{'message': 'Hello Surendra'}


## 2. A Tool-Calling Agent (ReAct pattern)

A more realistic graph: the LLM decides whether to call a tool; if it does, a `tools` node executes it and loops back to the LLM; otherwise the graph ends. This *reason → act → observe* loop is the core mechanism behind most LLM "agents."

**Heads up:** there's an incomplete piece further down (see the note near the graph-wiring cell) — worth spotting during a walkthrough.

In [2]:
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.prebuilt import ToolNode

### Define the tool, load the model, and bind them together

Same `@tool` pattern as the OpenAI notebook. `llm.bind_tools([...])` makes the tool available to the model on every subsequent call.

In [3]:
@tool
def calculate_sum(a: int, b: int) -> int:
    """Calculate the sum of two numbers."""
    return a + b

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY") 
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

In [5]:
llm_with_tools = llm.bind_tools(
    [calculate_sum]
)

### Graph state with message history

`messages: Annotated[list, add_messages]` — the `add_messages` reducer makes new messages get **appended** to the list rather than overwriting it. This is what lets the graph accumulate conversation/tool-call history across loop iterations instead of losing it on every node update.

In [6]:
from langgraph.graph.message import add_messages
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [7]:
tools = [calculate_sum]

tool_node = ToolNode(tools)

In [8]:
def call_llm(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

### `ToolNode`: a prebuilt node that executes tool calls

`ToolNode` inspects the last AI message for `tool_calls` and runs the matching Python function(s) automatically — this replaces the manual `tool.invoke(...)` step done by hand in the OpenAI notebook.

In [9]:
def should_continue(state: State):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END

### Conditional routing

`should_continue` inspects the latest message: if the model requested a tool call, route to the `tools` node; otherwise route to `END`. This function is the "decision point" wired into `add_conditional_edges` next.

In [10]:
graph = StateGraph(State)

graph.add_node("llm", call_llm)
graph.add_node("tools", tool_node)

graph.add_edge(START, "llm")

graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

graph.add_edge("tools", "llm")

app = graph.compile()

In [11]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [HumanMessage(content="What is 5 multiply 7?")]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is 5 multiply 7?
================================== Ai Message ==================================

The result of multiplying 5 by 7 is 35.


In [12]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [HumanMessage(content="What is 5 plus 7?")]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is 5 plus 7?
================================== Ai Message ==================================
Tool Calls:
  calculate_sum (call_g2fpywyw6HficqXh7A7DqvD0)
 Call ID: call_g2fpywyw6HficqXh7A7DqvD0
  Args:
    a: 5
    b: 7
================================= Tool Message =================================
Name: calculate_sum

12
================================== Ai Message ==================================

5 plus 7 is 12.


Tracing the loop for this input: `llm` sees the question and returns an `AIMessage` with a `tool_calls` entry for `calculate_sum(a=5, b=7)` → `should_continue` routes to `tools` → `ToolNode` runs the function and appends a `ToolMessage` with the result (`12`) → back to `llm`, which now has the tool result in its message history and returns a plain-text final answer → `should_continue` sees no `tool_calls` this time and routes to `END`. `pretty_print()`-ing every message in `result["messages"]` makes that whole trajectory visible.

## Recap

- **`StateGraph`**: nodes are plain functions, edges are control flow, `compile()` produces a runnable `app` with the same `.invoke()` interface as a single LLM call
- **State reducers**: `Annotated[list, add_messages]` makes state updates *append* instead of overwrite — required for accumulating history across loop iterations
- **`ToolNode`**: a prebuilt node that inspects the last `AIMessage.tool_calls` and executes the matching Python function(s) automatically
- **`add_conditional_edges`**: routes based on a function's return value — here, `should_continue` decides between looping back to `tools` or exiting to `END`
- Together these give the **ReAct loop**: `llm → (tool_calls?) → tools → llm → ... → END`